# 02 – Bronze Ingestion (Task B01)

**CommerceFlow 360** · Mục báo cáo: 3.3

Nạp 9 CSV Olist và `holidays_brazil.csv` vào DuckDB thành 10 bảng `bronze_*`:

- Giữ nguyên dữ liệu nguồn: đọc mọi cột dạng chuỗi, không đổi tên cột, không xóa dòng.
- Thêm 2 cột metadata `_ingested_at`, `_source_file`.
- Ghi bảng `ingestion_log` (số dòng CSV đếm độc lập, số dòng đã nạp, trạng thái).

Logic nằm ở `src/ingestion/bronze_ingestion.py` để notebook, `scripts/load_bronze.py` và pipeline đầy đủ dùng chung một cách nạp.

## 1. Chuẩn bị môi trường

In [1]:
# Trên Colab: clone repository và cài thư viện (bỏ qua khi chạy trong repository trên máy cá nhân)
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not Path('/content/CommerceFlow360').exists():
        !git clone -q https://github.com/huyenpiano1711/CommerceFlow360.git /content/CommerceFlow360
    %cd /content/CommerceFlow360
    %pip install -q -r requirements.txt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'dbt_project' / 'dbt_project.yml').is_file())
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.project import DUCKDB_PATH, TABLE_DIR, query
from src.ingestion.bronze_ingestion import RAW_FILES, run as run_bronze
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DUCKDB_PATH :', DUCKDB_PATH)

PROJECT_ROOT: D:\CommerceFlow360
DUCKDB_PATH : D:\CommerceFlow360\commerceflow.duckdb


## 2. Nạp Bronze và ghi nhật ký

In [2]:
bronze_log = run_bronze(PROJECT_ROOT, DUCKDB_PATH)
display(bronze_log)

,table_name,source_file,csv_rows,loaded_rows,column_count,status,ingested_at
0,bronze_customers,olist_customers_dataset.csv,99441,99441,5,PASS,2026-11-02 16:49:05.842651
1,bronze_geolocation,olist_geolocation_dataset.csv,1000163,1000163,5,PASS,2026-11-02 16:49:05.842651
2,bronze_order_items,olist_order_items_dataset.csv,112650,112650,7,PASS,2026-11-02 16:49:05.842651
3,bronze_order_payments,olist_order_payments_dataset.csv,103886,103886,5,PASS,2026-11-02 16:49:05.842651
4,bronze_order_reviews,olist_order_reviews_dataset.csv,99224,99224,7,PASS,2026-11-02 16:49:05.842651
5,bronze_orders,olist_orders_dataset.csv,99441,99441,8,PASS,2026-11-02 16:49:05.842651
6,bronze_products,olist_products_dataset.csv,32951,32951,9,PASS,2026-11-02 16:49:05.842651
7,bronze_sellers,olist_sellers_dataset.csv,3095,3095,4,PASS,2026-11-02 16:49:05.842651
8,bronze_product_category_name_translation,product_category_name_translation.csv,71,71,2,PASS,2026-11-02 16:49:05.842651
9,bronze_holidays,holidays_brazil.csv,27,27,4,PASS,2026-11-02 16:49:05.842651


## 3. Kiểm tra sau khi nạp

In [3]:
tables = query("select table_name from information_schema.tables where table_name like 'bronze_%' order by 1")
meta = query("""
    select count(*) filter (where _ingested_at is null or _source_file is null) as thieu_metadata
    from bronze_orders
""")
display(tables)
display(meta)
checks = {
    'đủ 10 bảng bronze_*': set(tables['table_name']) == set(RAW_FILES),
    'số dòng khớp CSV': bronze_log['status'].eq('PASS').all(),
    'đủ metadata': int(meta.iloc[0, 0]) == 0,
}
print(checks)
assert all(checks.values())

,table_name
0,bronze_customers
1,bronze_geolocation
2,bronze_holidays
3,bronze_order_items
4,bronze_order_payments
5,bronze_order_reviews
6,bronze_orders
7,bronze_product_category_name_translation
8,bronze_products
9,bronze_sellers


,thieu_metadata
0,0


{'đủ 10 bảng bronze_*': True, 'số dòng khớp CSV': np.True_, 'đủ metadata': True}


## 4. Lưu kết quả

Xuất nhật ký nạp cho báo cáo mục 3.3. Trên Colab có thể sao lưu file DuckDB lên Drive theo quy tắc `commerceflow_YYYYMMDD.duckdb` (đặt `SAO_LUU_DRIVE = True`).

In [4]:
TABLE_DIR.mkdir(parents=True, exist_ok=True)
bronze_log.to_csv(TABLE_DIR / 'Bang_3.3.1_NhatKyNapBronze.csv', index=False, encoding='utf-8-sig')
print('Đã xuất: reports/tables/Bang_3.3.1_NhatKyNapBronze.csv')

SAO_LUU_DRIVE = False
if IN_COLAB and SAO_LUU_DRIVE:
    import shutil, datetime
    from google.colab import drive
    drive.mount('/content/drive')
    backup_dir = Path('/content/drive/MyDrive/02_Warehouse_Backup')
    backup_dir.mkdir(parents=True, exist_ok=True)
    target = backup_dir / f"commerceflow_{datetime.date.today():%Y%m%d}.duckdb"
    shutil.copy(DUCKDB_PATH, target)
    print('Đã sao lưu:', target)

Đã xuất: reports/tables/Bang_3.3.1_NhatKyNapBronze.csv
